# Single Product Evidence Harness

Runs one product and highlights the **production-grade URL gate**.

Handoff rule for browser/scraping/coding teams:

```text
production_url_ready == True
production_url_status == "PRODUCTION_READY_EXACT_SCRAPABLE_BROWSER_URL"
needs_review == False
```

`product_url` is still the best discovered URL, but only rows passing the production gate should be treated as team-ready.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT

In [ ]:
from product_evidence_harness import (
    HarnessConfig,
    ProductEvidenceHarness,
    ProductQuery,
    SerpAPIConfig,
    ProductionURLGate,
    configure_logging,
)

configure_logging("INFO")

In [ ]:
product = ProductQuery(
    row_id="CO-ML-0001",
    main_text="PUT PRODUCT TEXT HERE",
    country_code="CO",
    ean="",  # optional; keep EAN/GTIN as text
    retailer_name="Mercado Libre",  # optional preferred-first evidence source
)
product

In [ ]:
config = HarnessConfig.from_env(PROJECT_ROOT / ".env")
serp_config = SerpAPIConfig.from_env(
    country_code=product.country_code,
    language_code=product.language_code or "es",
    env_file=PROJECT_ROOT / ".env",
)

harness = ProductEvidenceHarness(serp_config=serp_config, config=config)
trace = harness.run(product, return_trace=True)
match = trace.best_match

In [ ]:
production = ProductionURLGate().assess_url_in_state(trace.state, match.product_url or "")

summary = {
    "product_url": match.product_url,
    "verified_exact_url": match.verified_exact_url,
    "production_url_ready": production.production_ready if production else False,
    "production_url_status": production.status if production else "PRODUCT_URL_NOT_ASSESSED_OR_NO_SCORECARD",
    "browser_openable": production.browser_openable if production else False,
    "highly_scrapable": production.highly_scrapable if production else False,
    "exact_product_url_match": production.exact_product_match if production else False,
    "production_url_score": production.score if production else 0.0,
    "production_url_reasons": " | ".join(production.reasons) if production else "NO_SCORECARD_FOR_SELECTED_PRODUCT_URL",
    "needs_review": match.needs_review,
    "is_scrapable": match.is_scrapable,
    "url_decision_status": match.url_decision_status,
    "confidence": match.confidence,
}

pd.Series(summary).to_frame("value")

In [ ]:
handoff_ready = bool(
    production
    and production.production_ready
    and production.status == "PRODUCTION_READY_EXACT_SCRAPABLE_BROWSER_URL"
    and not match.needs_review
)

print("HANDOFF_READY_FOR_BROWSER_AND_SCRAPING_TEAM =", handoff_ready)
if not handoff_ready:
    print("This product_url is available but review-only / not production handoff-ready.")

In [ ]:
row_dir = Path(config.output_dir) / product.row_id
print("Row artifact directory:", row_dir)
if row_dir.exists():
    print("Files:")
    for p in sorted(row_dir.iterdir()):
        print("-", p.name)

final_row = row_dir / "final_row.csv"
if final_row.exists():
    display(pd.read_csv(final_row, dtype=str).T)

In [ ]:
coding_path = row_dir / "product_coding_input.json"
if coding_path.exists():
    payload = json.loads(coding_path.read_text(encoding="utf-8"))
    print(json.dumps({
        "selected_url": payload.get("selected_url"),
        "verified_exact_url": payload.get("verified_exact_url"),
        "quality_tier": payload.get("quality_tier"),
        "coding_readiness_status": payload.get("coding_readiness_status"),
        "review_flags": payload.get("review_flags"),
    }, indent=2, ensure_ascii=False))
else:
    print("product_coding_input.json not found")

In [ ]:
# Candidate-level production gate debugging
rows = []
gate = ProductionURLGate()
for card in trace.state.scorecards[:20]:
    assessment = gate.assess_card(card)
    rows.append({
        "url": card.candidate.url,
        "production_ready": assessment.production_ready,
        "status": assessment.status,
        "browser_openable": assessment.browser_openable,
        "highly_scrapable": assessment.highly_scrapable,
        "exact_product_url_match": assessment.exact_product_match,
        "reasons": " | ".join(assessment.reasons),
        "final_confidence": card.final_confidence,
    })

pd.DataFrame(rows)